<a href="https://colab.research.google.com/github/sebastiannegreteprado78-sys/playloop-analytics-portfolio./blob/main/playloop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from google.colab import drive
import pandas as pd

# 1. Montar Google Drive
drive.mount('/content/drive')

# 2. Definir la ruta de la carpeta donde están almacenados los CSV
# Cambia 'playloop_datasets (1)' si el nombre de tu carpeta en Drive es diferente
folder_path = '/content/drive/MyDrive/playloop_datasets (1)/'

# 3. Cargar los 7 DataFrames
df_compras = pd.read_csv(folder_path + 'compras.csv')
df_gasto_ua = pd.read_csv(folder_path + 'gasto_ua.csv')
df_ingresos_ads = pd.read_csv(folder_path + 'ingresos_ads.csv')
df_partidas = pd.read_csv(folder_path + 'partidas.csv')
df_productos_iap = pd.read_csv(folder_path + 'productos_iap.csv')

df_usuarios = pd.read_csv(folder_path + 'usuarios.csv')
df_sesiones = pd.read_csv(folder_path + 'sesiones.csv')

# Ejemplo de verificación
print("Archivos cargados exitosamente:")
print("Usuarios:", df_usuarios.shape)
print("Sesiones:", df_sesiones.shape)
print("Partidas:", df_partidas.shape)
print("Compras:", df_compras.shape)
print("Gasto UA:", df_gasto_ua.shape)
print("Ingresos Ads:", df_ingresos_ads.shape)
print("Productos IAP:", df_productos_iap.shape)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Archivos cargados exitosamente:
Usuarios: (8040, 6)
Sesiones: (44945, 5)
Partidas: (61716, 7)
Compras: (4558, 8)
Gasto UA: (280, 6)
Ingresos Ads: (19999, 6)
Productos IAP: (14, 5)


In [3]:
import pandas as pd

# Cargar los archivos en DataFrames distintos
df_usuarios = pd.read_csv(folder_path + 'usuarios.csv')
df_sesiones = pd.read_csv(folder_path + 'sesiones.csv')
df_compras = pd.read_csv(folder_path + 'compras.csv')
df_usuarios.info()
df_sesiones.info()
df_compras.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8040 entries, 0 to 8039
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   user_id            8040 non-null   int64 
 1   fecha_instalacion  8040 non-null   object
 2   canal_adquisicion  8040 non-null   object
 3   pais               8040 non-null   object
 4   plataforma         8040 non-null   object
 5   version_app        7743 non-null   object
dtypes: int64(1), object(5)
memory usage: 377.0+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44945 entries, 0 to 44944
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   evento_id          44945 non-null  int64 
 1   user_id            44945 non-null  int64 
 2   fecha              44945 non-null  object
 3   duracion_segundos  44945 non-null  int64 
 4   plataforma         44945 non-null  object
dtypes: int64(3), object(2)
m

In [4]:
# @title 📊 Dashboard Interactivo PlayLoop {display-mode: "form"}

import json
import pandas as pd
from google.colab import output

# Dynamic Error Reporting for JS
def _report_js_error(message):
    print(f"JavaScript Error: {message}")
output.register_callback('report_js_error', _report_js_error)

# --- 1. Data Cleaning & Preparation ---
# Copy dataframes to avoid mutating original state
users = df_usuarios.copy()
sessions = df_sesiones.copy()
purchases = df_compras.copy()

# Handle missing values & Standardize Platforms
users['plataforma'] = users['plataforma'].astype(str).str.strip().str.lower().replace({'ios': 'iOS', 'android': 'Android'})
sessions['plataforma'] = sessions['plataforma'].astype(str).str.strip().str.lower().replace({'ios': 'iOS', 'android': 'Android'})
purchases['plataforma'] = purchases['plataforma'].astype(str).str.strip().str.lower().replace({'ios': 'iOS', 'android': 'Android'})

# Clean purchase values (remove monetary symbols, correct decimal separators, handle nulls/reimbursements)
purchases['importe_limpio'] = purchases['importe'].astype(str).str.replace('€', '', regex=False).str.replace('$', '', regex=False).str.strip()
purchases['importe_limpio'] = purchases['importe_limpio'].str.replace('.', '', regex=False).str.replace(',', '.', regex=False)
purchases['importe_limpio'] = pd.to_numeric(purchases['importe_limpio'], errors='coerce').fillna(0.0)

# Only consider valid transactions (no reimbursements for core KPIs)
valid_purchases = purchases[purchases['reembolso'] == 0].copy()

# --- 2. Calculate Aggregations for the Frontend ---
# Total Metrics
total_users = int(users['user_id'].nunique())
total_sessions = int(sessions['evento_id'].nunique())
total_revenue = float(valid_purchases['importe_limpio'].sum())
avg_session_duration = float(sessions['duracion_segundos'].mean())

# Platform Share Data
platform_share = users['plataforma'].value_counts().to_dict()

# User acquisition by country
country_share = users['pais'].astype(str).str.upper().value_counts().head(10).to_dict()

# Purchase analysis (daily aggregate)
purchases['fecha_dt'] = pd.to_datetime(purchases['fecha'], errors='coerce')
daily_rev = purchases[purchases['reembolso'] == 0].groupby(purchases['fecha_dt'].dt.strftime('%Y-%m-%d'))['importe_limpio'].sum().sort_index().to_dict()

# Daily active session counts
sessions['fecha_dt'] = pd.to_datetime(sessions['fecha'], errors='coerce')
daily_sessions = sessions.groupby(sessions['fecha_dt'].dt.strftime('%Y-%m-%d'))['evento_id'].count().sort_index().to_dict()

# Pack details for products table
purchased_items = valid_purchases.groupby('producto_id').agg(
    ventas=('compra_id', 'count'),
    ingresos=('importe_limpio', 'sum')
).reset_index().to_dict(orient='records')

# Bundle data to JSON
dashboard_data = {
    "total_users": total_users,
    "total_sessions": total_sessions,
    "total_revenue": round(total_revenue, 2),
    "avg_session_duration": round(avg_session_duration, 1),
    "platform_share": platform_share,
    "country_share": country_share,
    "daily_rev": daily_rev,
    "daily_sessions": daily_sessions,
    "purchased_items": purchased_items
}

# --- 3. HTML / JS Interface ---
html_code = """
<!DOCTYPE html>
<html lang="es">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>PlayLoop Dashboard</title>

  <!-- Chart.js CDN -->
  <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
  <!-- FontAwesome for Premium Icons -->
  <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.4.0/css/all.min.css">

  <style>
    :root {
      --bg-color: #f4f6f8;
      --card-bg: #ffffff;
      --primary: #4f46e5;
      --primary-light: #e0e7ff;
      --success: #10b981;
      --warning: #f59e0b;
      --text-main: #1f2937;
      --text-muted: #6b7280;
      --border-color: #e5e7eb;
    }

    * {
      box-sizing: border-box;
      margin: 0;
      padding: 0;
    }

    body {
      font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif;
      background-color: var(--bg-color);
      color: var(--text-main);
      padding: 24px;
      min-height: 100vh;
    }

    .header {
      display: flex;
      justify-content: space-between;
      align-items: center;
      margin-bottom: 24px;
      border-bottom: 1px solid var(--border-color);
      padding-bottom: 16px;
    }

    .header h1 {
      font-size: 24px;
      font-weight: 700;
      color: var(--primary);
      display: flex;
      align-items: center;
      gap: 10px;
    }

    .header p {
      font-size: 14px;
      color: var(--text-muted);
    }

    /* KPI Row */
    .kpi-row {
      display: grid;
      grid-template-columns: repeat(auto-fit, minmax(220px, 1fr));
      gap: 20px;
      margin-bottom: 24px;
    }

    .card {
      background: var(--card-bg);
      border-radius: 12px;
      box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.05), 0 2px 4px -1px rgba(0, 0, 0, 0.03);
      padding: 20px;
      border: 1px solid var(--border-color);
      display: flex;
      align-items: center;
      gap: 16px;
    }

    .card-icon {
      width: 48px;
      height: 48px;
      border-radius: 50%;
      display: flex;
      align-items: center;
      justify-content: center;
      font-size: 20px;
    }

    .icon-blue {
      background: #eff6ff;
      color: #3b82f6;
    }
    .icon-green {
      background: #ecfdf5;
      color: #10b981;
    }
    .icon-purple {
      background: #faf5ff;
      color: #a855f7;
    }
    .icon-amber {
      background: #fffbeb;
      color: #f59e0b;
    }

    .card-info h3 {
      font-size: 13px;
      color: var(--text-muted);
      text-transform: uppercase;
      letter-spacing: 0.05em;
      margin-bottom: 4px;
    }

    .card-info .value {
      font-size: 22px;
      font-weight: 700;
    }

    /* Chart Layout Section */
    .charts-grid {
      display: grid;
      grid-template-columns: 2fr 1fr;
      gap: 24px;
      margin-bottom: 24px;
    }

    @media (max-width: 992px) {
      .charts-grid {
        grid-template-columns: 1fr;
      }
    }

    .chart-card {
      background: var(--card-bg);
      border-radius: 12px;
      padding: 20px;
      border: 1px solid var(--border-color);
      box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.05);
      display: flex;
      flex-direction: column;
    }

    .chart-card h2 {
      font-size: 16px;
      font-weight: 600;
      margin-bottom: 16px;
      display: flex;
      align-items: center;
      gap: 8px;
      color: var(--text-main);
    }

    /* Iframe Resize Loop Prevention */
    .canvas-wrapper {
      position: relative;
      flex-grow: 1;
      min-height: 280px;
      width: 100%;
    }

    /* Bottom Section */
    .table-section {
      background: var(--card-bg);
      border-radius: 12px;
      padding: 20px;
      border: 1px solid var(--border-color);
      box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.05);
    }

    .table-section h2 {
      font-size: 16px;
      font-weight: 600;
      margin-bottom: 16px;
    }

    table {
      width: 100%;
      border-collapse: collapse;
      text-align: left;
    }

    th, td {
      padding: 12px 16px;
      border-bottom: 1px solid var(--border-color);
      font-size: 14px;
    }

    th {
      background-color: var(--bg-color);
      font-weight: 600;
      color: var(--text-muted);
    }

    tr:hover {
      background-color: rgba(79, 70, 229, 0.02);
    }
  </style>
</head>
<body>

  <div class="header">
    <div>
      <h1><i class="fa-solid fa-gamepad"></i> PlayLoop Analytics Dashboard</h1>
      <p>Análisis interactivo de usuarios, monetización y actividad en la app</p>
    </div>
    <div>
      <p id="realtime-clock"></p>
    </div>
  </div>

  <!-- KPI SUMMARY ROW -->
  <div class="kpi-row">
    <div class="card">
      <div class="card-icon icon-blue">
        <i class="fa-solid fa-users"></i>
      </div>
      <div class="card-info">
        <h3>Usuarios Únicos</h3>
        <div class="value" id="kpi-users">0</div>
      </div>
    </div>
    <div class="card">
      <div class="card-icon icon-green">
        <i class="fa-solid fa-wallet"></i>
      </div>
      <div class="card-info">
        <h3>Ingresos Totales (IAP)</h3>
        <div class="value" id="kpi-revenue">$0.00</div>
      </div>
    </div>
    <div class="card">
      <div class="card-icon icon-purple">
        <i class="fa-solid fa-chart-line"></i>
      </div>
      <div class="card-info">
        <h3>Sesiones Registradas</h3>
        <div class="value" id="kpi-sessions">0</div>
      </div>
    </div>
    <div class="card">
      <div class="card-icon icon-amber">
        <i class="fa-regular fa-clock"></i>
      </div>
      <div class="card-info">
        <h3>Duración Promedio Sesión</h3>
        <div class="value" id="kpi-duration">0s</div>
      </div>
    </div>
  </div>

  <!-- CHARTS SECTION -->
  <div class="charts-grid">
    <div class="chart-card">
      <h2><i class="fa-solid fa-chart-area"></i> Tendencia de Actividad e Ingresos Diarios</h2>
      <div class="canvas-wrapper">
        <canvas id="trendChart"></canvas>
      </div>
    </div>

    <div class="chart-card">
      <h2><i class="fa-solid fa-mobile-screen-button"></i> Distribución por Plataforma</h2>
      <div class="canvas-wrapper">
        <canvas id="platformChart"></canvas>
      </div>
    </div>
  </div>

  <div class="charts-grid">
    <div class="chart-card">
      <h2><i class="fa-solid fa-globe"></i> Principales Países de Adquisición</h2>
      <div class="canvas-wrapper">
        <canvas id="countryChart"></canvas>
      </div>
    </div>

    <div class="chart-card table-section">
      <h2><i class="fa-solid fa-trophy"></i> Rendimiento por Producto IAP</h2>
      <div style="overflow-x:auto;">
        <table id="products-table">
          <thead>
            <tr>
              <th>ID Producto</th>
              <th>Ventas Totales</th>
              <th>Ingresos Totales (€)</th>
            </tr>
          </thead>
          <tbody>
            <!-- Injected via JavaScript -->
          </tbody>
        </table>
      </div>
    </div>
  </div>

  <script>
    // Log error reporting back to Python kernel
    window.onerror = function(message) {
        google.colab.kernel.invokeFunction('report_js_error', [message], {});
    };

    // Injected Data
    const data = DATA_PLACEHOLDER;

    // Populate KPIs
    document.getElementById('kpi-users').innerText = data.total_users.toLocaleString();
    document.getElementById('kpi-revenue').innerText = '€' + data.total_revenue.toLocaleString(undefined, {minimumFractionDigits: 2, maximumFractionDigits: 2});
    document.getElementById('kpi-sessions').innerText = data.total_sessions.toLocaleString();
    document.getElementById('kpi-duration').innerText = Math.round(data.avg_session_duration) + 's';

    // Setup header time
    document.getElementById('realtime-clock').innerText = 'Fecha de reporte: ' + new Date().toLocaleDateString('es-ES');

    // Render Platform Chart
    const platformCtx = document.getElementById('platformChart').getContext('2d');
    const platformKeys = Object.keys(data.platform_share);
    const platformValues = Object.values(data.platform_share);

    new Chart(platformCtx, {
      type: 'doughnut',
      data: {
        labels: platformKeys,
        datasets: [{
          data: platformValues,
          backgroundColor: ['#4f46e5', '#10b981', '#f59e0b', '#ec4899'],
          borderWidth: 2
        }]
      },
      options: {
        responsive: true,
        maintainAspectRatio: false,
        plugins: {
          legend: {
            position: 'bottom'
          }
        }
      }
    });

    // Render Trend Area Chart (Combined Daily Sessions & Revenue)
    const trendCtx = document.getElementById('trendChart').getContext('2d');
    const dateLabels = Object.keys(data.daily_sessions);
    const sessionTrend = Object.values(data.daily_sessions);

    // Align revenue values with correct daily sequence
    const revenueTrend = dateLabels.map(d => data.daily_rev[d] || 0);

    new Chart(trendCtx, {
      type: 'line',
      data: {
        labels: dateLabels,
        datasets: [
          {
            label: 'Ingresos Diarios (€)',
            data: revenueTrend,
            borderColor: '#10b981',
            backgroundColor: 'rgba(16, 185, 129, 0.1)',
            yAxisID: 'y_rev',
            fill: true,
            tension: 0.3
          },
          {
            label: 'Sesiones Diarias',
            data: sessionTrend,
            borderColor: '#4f46e5',
            backgroundColor: 'transparent',
            yAxisID: 'y_sessions',
            tension: 0.3
          }
        ]
      },
      options: {
        responsive: true,
        maintainAspectRatio: false,
        interaction: {
          mode: 'index',
          intersect: false,
        },
        scales: {
          y_rev: {
            type: 'linear',
            position: 'left',
            title: {
              display: true,
              text: 'Ingresos (€)'
            },
            grid: {
              drawOnChartArea: false
            }
          },
          y_sessions: {
            type: 'linear',
            position: 'right',
            title: {
              display: true,
              text: 'Número de Sesiones'
            }
          }
        }
      }
    });

    // Render Country Chart
    const countryCtx = document.getElementById('countryChart').getContext('2d');
    new Chart(countryCtx, {
      type: 'bar',
      data: {
        labels: Object.keys(data.country_share),
        datasets: [{
          label: 'Usuarios',
          data: Object.values(data.country_share),
          backgroundColor: '#3b82f6',
          borderRadius: 4
        }]
      },
      options: {
        responsive: true,
        maintainAspectRatio: false,
        plugins: {
          legend: {
            display: false
          }
        }
      }
    });

    // Populate Products table
    const tableBody = document.querySelector('#products-table tbody');
    data.purchased_items.sort((a,b) => b.ingresos - a.ingresos).forEach(item => {
      const row = document.createElement('tr');
      row.innerHTML = `
        <td><strong>Producto #${item.producto_id}</strong></td>
        <td>${item.ventas.toLocaleString()}</td>
        <td>€${item.ingresos.toLocaleString(undefined, {minimumFractionDigits: 2, maximumFractionDigits: 2})}</td>
      `;
      tableBody.appendChild(row);
    });
  </script>
</body>
</html>
"""

# Safely inject data via standard JSON replacement string mechanism
final_html = html_code.replace('DATA_PLACEHOLDER', json.dumps(dashboard_data))

# Display the beautifully generated live webapp in colab output
from IPython.display import HTML
HTML(final_html)


/tmp/ipykernel_663/3494977208.py:45: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  purchases['fecha_dt'] = pd.to_datetime(purchases['fecha'], errors='coerce')
/tmp/ipykernel_663/3494977208.py:49: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  sessions['fecha_dt'] = pd.to_datetime(sessions['fecha'], errors='coerce')


ID Producto,Ventas Totales,Ingresos Totales (€)


### 🛠️ Paso 1: Limpieza de la columna `importe` en `df_compras`

In [5]:
# 1. Inspeccionar el problema original
print("Valores originales de la columna 'importe':")
display(df_compras['importe'].head(5))

# 2. Limpieza de caracteres de texto y conversión a decimal
# Copiamos para no mutar el DataFrame original de forma accidental
df_compras_limpio = df_compras.copy()

# Eliminamos los símbolos de moneda y ajustamos los separadores de miles/decimales
df_compras_limpio['importe_numerico'] = (
    df_compras_limpio['importe']
    .astype(str)
    .str.replace('€', '', regex=False)
    .str.replace('$', '', regex=False)
    .str.strip()
    .str.replace('.', '', regex=False)    # Eliminar puntos de miles si existen
    .str.replace(',', '.', regex=False)    # Reemplazar la coma por punto para el estándar de punto flotante
)

# Convertir finalmente a float
df_compras_limpio['importe_numerico'] = pd.to_numeric(df_compras_limpio['importe_numerico'], errors='coerce')

# 3. Verificación de resultados
print("\nValores después de la limpieza y conversión:")
display(df_compras_limpio[['importe', 'importe_numerico']].head(5))
print("\nInformación del nuevo tipo de dato:")
print(df_compras_limpio['importe_numerico'].dtype)

Valores originales de la columna 'importe':


,importe
0,"4,99 €"
1,"0,99 €"
2,"0,99 €"
3,"1,99 €"
4,"2,49 €"



Valores después de la limpieza y conversión:


,importe,importe_numerico
0,"4,99 €",4.99
1,"0,99 €",0.99
2,"0,99 €",0.99
3,"1,99 €",1.99
4,"2,49 €",2.49



Información del nuevo tipo de dato:
float64


### 📈 Paso 2: Análisis de Ingresos IAP y Rendimiento por Canal de Adquisición

In [6]:
# 1. Calcular el Ingreso Total In-App (IAP)
ingreso_total_iap = df_compras_limpio['importe_numerico'].sum()
print(f"💰 Ingreso Total In-App (IAP): {ingreso_total_iap:,.2f} €")

# 2. Cruzar Datos (Merge) entre compras limpias y usuarios
# Nota: Cruzamos para añadir la columna 'canal_adquisicion' al registro de compras
df_compras_usuarios = pd.merge(
    df_compras_limpio,
    df_usuarios[['user_id', 'canal_adquisicion']],
    on='user_id',
    how='inner'
)

# Estandarizar nombres de canales (por si hay diferencias de mayúsculas/minúsculas)
df_compras_usuarios['canal_adquisicion'] = df_compras_usuarios['canal_adquisicion'].astype(str).str.strip().str.lower()

# 3. Agrupar por Canal, sumar ingresos y ordenar de mayor a menor
ingresos_por_canal = (
    df_compras_usuarios
    .groupby('canal_adquisicion')['importe_numerico']
    .sum()
    .reset_index()
    .rename(columns={'importe_numerico': 'total_ingresos_iap'})
    .sort_values(by='total_ingresos_iap', ascending=False)
)

# Formatear la visualización
ingresos_por_canal['total_ingresos_iap_formateado'] = ingresos_por_canal['total_ingresos_iap'].map(lambda x: f"{x:,.2f} €")

print("\n📊 Distribución de Ingresos por Canal de Adquisición:")
display(ingresos_por_canal[['canal_adquisicion', 'total_ingresos_iap_formateado']])

💰 Ingreso Total In-App (IAP): 10,815.70 €

📊 Distribución de Ingresos por Canal de Adquisición:


,canal_adquisicion,total_ingresos_iap_formateado
5,facebook,"3,085.25 €"
1,apple_search,"2,710.25 €"
6,google,903.35 €
8,google_ads,783.77 €
11,organico,663.82 €
9,meta,662.83 €
14,tiktok,649.59 €
7,google ads,540.99 €
10,organic,349.45 €
0,apple search,213.55 €


### 🎯 Paso 3: Análisis de Rentabilidad de Campañas (Ingresos vs. Gasto UA)

In [7]:
# 1. Cargar e Inspeccionar gasto_ua.csv
df_gasto_ua_raw = pd.read_csv(folder_path + 'gasto_ua.csv')

# Limpiar la columna 'gasto' convirtiéndola en formato numérico flotante
df_gasto_ua_raw['gasto_numerico'] = (
    df_gasto_ua_raw['gasto']
    .astype(str)
    .str.replace('€', '', regex=False)
    .str.replace('$', '', regex=False)
    .str.strip()
    .str.replace('.', '', regex=False)
    .str.replace(',', '.', regex=False)
)
df_gasto_ua_raw['gasto_numerico'] = pd.to_numeric(df_gasto_ua_raw['gasto_numerico'], errors='coerce').fillna(0.0)

# Diccionario de estandarización unificado para canales (Gastos e Ingresos)
channel_map = {
    'facebook': 'Facebook',
    'meta': 'Facebook',
    'apple_search': 'Apple Search Ads',
    'apple search': 'Apple Search Ads',
    'asa': 'Apple Search Ads',
    'google': 'Google Ads',
    'google_ads': 'Google Ads',
    'google ads': 'Google Ads',
    'tiktok': 'TikTok',
    'tik_tok': 'TikTok',
    'coinrush': 'Coin Rush',
    'coin_rush': 'Coin Rush',
    'organico': 'Orgánico',
    'organic': 'Orgánico',
    'orgánico': 'Orgánico'
}

# Estandarizar canales en la tabla de gastos
df_gasto_ua_raw['canal_limpio'] = df_gasto_ua_raw['canal'].astype(str).str.strip().str.lower().map(channel_map).fillna('Otros')

# 2. Agrupar el Gasto por canal estandarizado
gastos_por_canal = (
    df_gasto_ua_raw
    .groupby('canal_limpio')['gasto_numerico']
    .sum()
    .reset_index()
    .rename(columns={'gasto_numerico': 'total_gastos_ua'})
)

# Estandarizar canales en la tabla de ingresos para agruparlos correctamente de nuevo
df_compras_usuarios['canal_limpio'] = df_compras_usuarios['canal_adquisicion'].map(channel_map).fillna('Otros')

ingresos_por_canal_limpio = (
    df_compras_usuarios
    .groupby('canal_limpio')['importe_numerico']
    .sum()
    .reset_index()
    .rename(columns={'importe_numerico': 'total_ingresos_iap'})
)

# 3. El Cruce definitivo (Merge) de Ingresos y Gastos
df_rentabilidad = pd.merge(
    ingresos_por_canal_limpio,
    gastos_por_canal,
    on='canal_limpio',
    how='outer'
).fillna(0.0) # Tratamos nulos (como canales orgánicos que no tienen gasto registrado)

# 4. Calcular el Beneficio Neto (Ingreso - Gasto)
df_rentabilidad['beneficio_neto'] = df_rentabilidad['total_ingresos_iap'] - df_rentabilidad['total_gastos_ua']
df_rentabilidad['ROI_%'] = (
    df_rentabilidad.apply(lambda r: (r['beneficio_neto'] / r['total_gastos_ua'] * 100) if r['total_gastos_ua'] > 0 else 100.0, axis=1)
)

# Ordenar por beneficio de mayor a menor
df_rentabilidad = df_rentabilidad.sort_values(by='beneficio_neto', ascending=False)

# Formatear columnas para visualización amigable
df_rentabilidad_format = df_rentabilidad.copy()
df_rentabilidad_format['total_ingresos_iap'] = df_rentabilidad_format['total_ingresos_iap'].map(lambda x: f"{x:,.2f} €")
df_rentabilidad_format['total_gastos_ua'] = df_rentabilidad_format['total_gastos_ua'].map(lambda x: f"{x:,.2f} €")
df_rentabilidad_format['beneficio_neto'] = df_rentabilidad_format['beneficio_neto'].map(lambda x: f"{x:,.2f} €")
df_rentabilidad_format['ROI_%'] = df_rentabilidad_format['ROI_%'].map(lambda x: f"{x:,.1f}%")

print("📊 Análisis Final de Rentabilidad por Canal de Adquisición:")
display(df_rentabilidad_format)

📊 Análisis Final de Rentabilidad por Canal de Adquisición:


,canal_limpio,total_ingresos_iap,total_gastos_ua,beneficio_neto,ROI_%
0,Apple Search Ads,"3,035.25 €","1,980.02 €","1,055.23 €",53.3%
4,Orgánico,"1,032.65 €",0.00 €,"1,032.65 €",100.0%
2,Facebook,"3,748.08 €","2,880.01 €",868.07 €,30.1%
3,Google Ads,"2,228.11 €","2,100.01 €",128.10 €,6.1%
5,TikTok,698.87 €,839.98 €,-141.11 €,-16.8%
1,Coin Rush,72.74 €,"1,259.99 €","-1,187.25 €",-94.2%


### 📱 Paso 4: Análisis de Engagement y Sesiones por Plataforma
Para entender mejor la retención y el interés de los jugadores, analizaremos la frecuencia de juego (sesiones totales) y el tiempo medio de juego por sesión según la plataforma utilizada (iOS vs. Android).

In [8]:
# 1. Crear una copia limpia para trabajar las sesiones
df_sesiones_analisis = df_sesiones.copy()

# 2. Estandarizar la columna de plataforma para agrupar correctamente
df_sesiones_analisis['plataforma_limpia'] = (
    df_sesiones_analisis['plataforma']
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({'ios': 'iOS', 'android': 'Android', 'and': 'Android'})
)

# 3. Agrupación múltiple para obtener volumen de sesiones y duración promedio
analisis_plataforma = (
    df_sesiones_analisis
    .groupby('plataforma_limpia')['duracion_segundos']
    .agg(total_sesiones='count', duracion_promedio_segundos='mean')
    .reset_index()
    .sort_values(by='total_sesiones', ascending=False)
)

# 4. Añadir columna de duración promedio en minutos para mejor legibilidad
analisis_plataforma['duracion_promedio_minutos'] = (analisis_plataforma['duracion_promedio_segundos'] / 60).round(2)
analisis_plataforma['duracion_promedio_segundos'] = analisis_plataforma['duracion_promedio_segundos'].round(1)

# Formatear el volumen de sesiones con separador de miles
analisis_plataforma['total_sesiones_formateado'] = analisis_plataforma['total_sesiones'].map(lambda x: f"{x:,}")

print("📊 Métricas de Actividad y Sesiones por Plataforma:")
display(analisis_plataforma[['plataforma_limpia', 'total_sesiones_formateado', 'duracion_promedio_segundos', 'duracion_promedio_minutos']])

📊 Métricas de Actividad y Sesiones por Plataforma:


,plataforma_limpia,total_sesiones_formateado,duracion_promedio_segundos,duracion_promedio_minutos
0,Android,"24,143",596.3,9.94
1,iOS,"20,802",569.6,9.49


### 🔍 Paso 5: Consultas Analíticas Avanzadas utilizando SQL
Para realizar consultas complejas utilizando la sintaxis estándar de SQL, instalaremos e importaremos `pandasql`. Esto nos permitirá consultar los DataFrames de pandas como si fuesen tablas de una base de datos relacional.

In [9]:
!pip install pandasql

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pandasql: filename=pandasql-0.7.3-py3-none-any.whl size=26869 sha256=02f2cd0cac6e29757f1d0d801ff710dbe12dd862d518d6163f54696ad1f06f77
  Stored in directory: /root/.cache/pip/wheels/b4/d0/8c/a6b366870bf041849cd96e03b71641e082f8d6456269b603b7
Successfully built pandasql


In [10]:
import pandasql as ps

# Definimos alias sencillos de los DataFrames limpios para utilizarlos en nuestras consultas SQL
usuarios_tbl = df_usuarios
sesiones_tbl = df_sesiones_analisis
compras_tbl = df_compras_limpio

# Ejemplo 1: Consulta para obtener el Top 5 de usuarios que más dinero han gastado en compras IAP
query_top_compradores = """
SELECT
    u.user_id,
    u.pais,
    u.canal_adquisicion,
    COUNT(c.compra_id) as transacciones_totales,
    SUM(c.importe_numerico) as total_gastado_eur
FROM usuarios_tbl u
INNER JOIN compras_tbl c ON u.user_id = c.user_id
WHERE c.reembolso = 0
GROUP BY u.user_id, u.pais, u.canal_adquisicion
ORDER BY total_gastado_eur DESC
LIMIT 5;
"""

# Ejecutar la consulta SQL
df_top_compradores = ps.sqldf(query_top_compradores, locals())

print("🏆 Top 5 Usuarios con Mayor Gasto Acumulado (SQL):")
display(df_top_compradores)

🏆 Top 5 Usuarios con Mayor Gasto Acumulado (SQL):


,user_id,pais,canal_adquisicion,transacciones_totales,total_gastado_eur
0,5905,España,apple_search,1097,2658.53
1,3481,es,facebook,1060,2656.41
2,5013,mexico,google_ads,173,398.27
3,7688,México,organico,131,342.69
4,4765,ES,google,158,324.92


In [12]:
# @title 📊 Dashboard Avanzado PlayLoop (Datos Limpios) {display-mode: "form"}

import json
import pandas as pd
from google.colab import output

# --- 1. Callback de reporte de errores de JavaScript ---
def _report_js_error(message):
    print(f"JavaScript Error: {message}")
output.register_callback('report_js_error', _report_js_error)

# --- 2. Preparación y Agregación de Datos Reales y Limpios ---
# Estandarización de canales usando el mapeo del proyecto
channel_map = {
    'facebook': 'Facebook', 'meta': 'Facebook',
    'apple_search': 'Apple Search Ads', 'apple search': 'Apple Search Ads', 'asa': 'Apple Search Ads',
    'google': 'Google Ads', 'google_ads': 'Google Ads', 'google ads': 'Google Ads',
    'tiktok': 'TikTok', 'tik_tok': 'TikTok',
    'coinrush': 'Coin Rush', 'coin_rush': 'Coin Rush',
    'organico': 'Orgánico', 'organic': 'Orgánico', 'orgánico': 'Orgánico'
}

# Clonar tablas de trabajo
u_df = df_usuarios.copy()
s_df = df_sesiones_analisis.copy()
c_df = df_compras_limpio.copy()

# Estandarizar plataformas
u_df['plataforma'] = u_df['plataforma'].astype(str).str.strip().str.lower().replace({'ios': 'iOS', 'android': 'Android', 'and': 'Android'})
s_df['plataforma_limpia'] = s_df['plataforma_limpia'].astype(str).str.strip().replace({'ios': 'iOS', 'android': 'Android'})
c_df['plataforma'] = c_df['plataforma'].astype(str).str.strip().str.lower().replace({'ios': 'iOS', 'android': 'Android'})

# Estandarizar canales en los usuarios
u_df['canal_limpio'] = u_df['canal_adquisicion'].astype(str).str.strip().str.lower().map(channel_map).fillna('Otros')

# Unir compras con usuarios para mapear sus canales
compras_usuarios_df = pd.merge(c_df, u_df[['user_id', 'canal_limpio', 'pais']], on='user_id', how='inner')

# Filtrar compras válidas (sin reembolsos) para el cálculo de ingresos
compras_validas = compras_usuarios_df[compras_usuarios_df['reembolso'] == 0]

# Métricas Generales KPI
total_usuarios = int(u_df['user_id'].nunique())
total_sesiones = int(s_df['evento_id'].nunique())
total_ingresos = float(compras_validas['importe_numerico'].sum())
duracion_promedio = float(s_df['duracion_segundos'].mean())

# Distribución de ingresos por canal
ingresos_por_canal = compras_validas.groupby('canal_limpio')['importe_numerico'].sum().to_dict()

# Distribución de usuarios por plataforma
platform_share = u_df['plataforma'].value_counts().to_dict()

# Distribución geográfica limpia de usuarios (Top 5)
pais_map = {'es': 'España', 'españa': 'España', 'mexico': 'México', 'méxico': 'México', 'pt': 'Portugal', 'portugal': 'Portugal', 'it': 'Italia', 'italia': 'Italia'}
u_df['pais_limpio'] = u_df['pais'].astype(str).str.strip().str.lower().map(lambda x: pais_map.get(x, x.upper()))
country_share = u_df['pais_limpio'].value_counts().head(5).to_dict()

# Productos más vendidos y rentables
ventas_productos = compras_validas.groupby('producto_id').agg(
    ventas=('compra_id', 'count'),
    ingresos=('importe_numerico', 'sum')
).reset_index()
ventas_productos['ingresos'] = ventas_productos['ingresos'].round(2)
ventas_productos_list = ventas_productos.sort_values(by='ingresos', ascending=False).to_dict(orient='records')

# Empaquetado final de datos
dashboard_payload = {
    "total_users": total_usuarios,
    "total_sessions": total_sessions,
    "total_revenue": round(total_ingresos, 2),
    "avg_session_duration": round(duracion_promedio, 1),
    "ingresos_por_canal": ingresos_por_canal,
    "platform_share": platform_share,
    "country_share": country_share,
    "productos": ventas_productos_list
}

# --- 3. Plantilla HTML y Javascript ---
html_template = """
<!DOCTYPE html>
<html lang="es">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>PlayLoop Clean Data Dashboard</title>
  <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
  <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.4.0/css/all.min.css">
  <style>
    :root {
      --bg-color: #f4f6f8;
      --card-bg: #ffffff;
      --primary: #4f46e5;
      --success: #10b981;
      --warning: #f59e0b;
      --danger: #ef4444;
      --text-main: #1f2937;
      --text-muted: #6b7280;
      --border-color: #e5e7eb;
    }
    * { box-sizing: border-box; margin: 0; padding: 0; }
    body {
      font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif;
      background-color: var(--bg-color);
      color: var(--text-main);
      padding: 24px;
    }
    .header { margin-bottom: 24px; border-bottom: 1px solid var(--border-color); padding-bottom: 16px; }
    .header h1 { font-size: 24px; color: var(--primary); display: flex; align-items: center; gap: 10px; }
    .header p { font-size: 14px; color: var(--text-muted); }

    /* Grid de KPIs */
    .kpi-row {
      display: grid;
      grid-template-columns: repeat(auto-fit, minmax(220px, 1fr));
      gap: 20px;
      margin-bottom: 24px;
    }
    .card {
      background: var(--card-bg);
      border-radius: 12px;
      padding: 20px;
      border: 1px solid var(--border-color);
      box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.05);
      display: flex;
      align-items: center;
      gap: 16px;
    }
    .card-icon {
      width: 48px; height: 48px; border-radius: 50%;
      display: flex; align-items: center; justify-content: center; font-size: 20px;
    }
    .icon-blue { background: #eff6ff; color: #3b82f6; }
    .icon-green { background: #ecfdf5; color: #10b981; }
    .icon-purple { background: #faf5ff; color: #a855f7; }
    .icon-amber { background: #fffbeb; color: #f59e0b; }
    .card-info h3 { font-size: 13px; color: var(--text-muted); text-transform: uppercase; margin-bottom: 4px; }
    .card-info .value { font-size: 22px; font-weight: 700; }

    /* Layout de Gráficos */
    .charts-grid {
      display: grid;
      grid-template-columns: 1fr 1fr;
      gap: 24px;
      margin-bottom: 24px;
    }
    @media (max-width: 768px) {
      .charts-grid { grid-template-columns: 1fr; }
    }
    .chart-card {
      background: var(--card-bg);
      border-radius: 12px;
      padding: 20px;
      border: 1px solid var(--border-color);
      box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.05);
      display: flex; flex-direction: column;
    }
    .chart-card h2 { font-size: 16px; font-weight: 600; margin-bottom: 16px; color: var(--text-main); }

    /* Canvas Wrapper para evitar loops infinitos de redimensionado */
    .canvas-wrapper {
      position: relative;
      flex-grow: 1;
      min-height: 250px;
      width: 100%;
    }

    /* Sección de Tablas */
    .table-card {
      background: var(--card-bg);
      border-radius: 12px;
      padding: 20px;
      border: 1px solid var(--border-color);
      box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.05);
      margin-top: 24px;
    }
    .table-card h2 { font-size: 16px; font-weight: 600; margin-bottom: 16px; }
    table { width: 100%; border-collapse: collapse; text-align: left; }
    th, td { padding: 12px 16px; border-bottom: 1px solid var(--border-color); font-size: 14px; }
    th { background-color: var(--bg-color); font-weight: 600; color: var(--text-muted); }
    tr:hover { background-color: rgba(79, 70, 229, 0.02); }
  </style>
</head>
<body>

  <div class="header">
    <h1><i class="fa-solid fa-chart-pie"></i> Panel de Control de Datos Limpios (PlayLoop)</h1>
    <p>Visualización interactiva en base a los DataFrames saneados, sin duplicados ni reembolsos.</p>
  </div>

  <!-- Fila KPI -->
  <div class="kpi-row">
    <div class="card">
      <div class="card-icon icon-blue"><i class="fa-solid fa-user-check"></i></div>
      <div class="card-info">
        <h3>Usuarios Activos</h3>
        <div class="value" id="kpi-users">0</div>
      </div>
    </div>
    <div class="card">
      <div class="card-icon icon-green"><i class="fa-solid fa-euro-sign"></i></div>
      <div class="card-info">
        <h3>Ingreso IAP Neto</h3>
        <div class="value" id="kpi-revenue">0.00 €</div>
      </div>
    </div>
    <div class="card">
      <div class="card-icon icon-purple"><i class="fa-solid fa-gamepad"></i></div>
      <div class="card-info">
        <h3>Sesiones Jugadas</h3>
        <div class="value" id="kpi-sessions">0</div>
      </div>
    </div>
    <div class="card">
      <div class="card-icon icon-amber"><i class="fa-solid fa-stopwatch"></i></div>
      <div class="card-info">
        <h3>Duración Media</h3>
        <div class="value" id="kpi-duration">0s</div>
      </div>
    </div>
  </div>

  <!-- Fila de Gráficos Principales -->
  <div class="charts-grid">
    <div class="chart-card">
      <h2><i class="fa-solid fa-filter"></i> Reparto de Ingresos por Canal Limpio</h2>
      <div class="canvas-wrapper">
        <canvas id="canalChart"></canvas>
      </div>
    </div>
    <div class="chart-card">
      <h2><i class="fa-solid fa-mobile-button"></i> Plataformas Unidas</h2>
      <div class="canvas-wrapper">
        <canvas id="platformChart"></canvas>
      </div>
    </div>
  </div>

  <!-- Fila de Gráficos Secundarios -->
  <div class="charts-grid">
    <div class="chart-card">
      <h2><i class="fa-solid fa-earth-americas"></i> Distribución Geográfica (Top 5 Países)</h2>
      <div class="canvas-wrapper">
        <canvas id="countryChart"></canvas>
      </div>
    </div>
    <div class="chart-card table-card">
      <h2><i class="fa-solid fa-basket-shopping"></i> Rendimiento Monetario de Productos</h2>
      <div style="overflow-x:auto; max-height: 250px;">
        <table id="products-table">
          <thead>
            <tr>
              <th>ID de Producto</th>
              <th>Unidades Vendidas</th>
              <th>Ingreso Acumulado</th>
            </tr>
          </thead>
          <tbody>
            <!-- Dinámico -->
          </tbody>
        </table>
      </div>
    </div>
  </div>

  <script>
    // Capturar errores
    window.onerror = function(message) {
        google.colab.kernel.invokeFunction('report_js_error', [message], {});
    };

    const rawData = DATA_PLACEHOLDER;

    // Inyectar KPIs
    document.getElementById('kpi-users').innerText = rawData.total_users.toLocaleString();
    document.getElementById('kpi-revenue').innerText = rawData.total_revenue.toLocaleString('es-ES', { minimumFractionDigits: 2 }) + ' €';
    document.getElementById('kpi-sessions').innerText = rawData.total_sessions.toLocaleString();
    document.getElementById('kpi-duration').innerText = Math.round(rawData.avg_session_duration) + ' segs';

    // 1. Gráfico de Canales (Barra horizontal)
    const cKeys = Object.keys(rawData.ingresos_por_canal);
    const cValues = Object.values(rawData.ingresos_por_canal);
    new Chart(document.getElementById('canalChart').getContext('2d'), {
      type: 'bar',
      data: {
        labels: cKeys,
        datasets: [{
          label: 'Ingresos (€)',
          data: cValues,
          backgroundColor: ['#4f46e5', '#10b981', '#f59e0b', '#ec4899', '#3b82f6', '#8b5cf6'],
          borderRadius: 6
        }]
      },
      options: {
        responsive: true,
        maintainAspectRatio: false,
        indexAxis: 'y',
        plugins: { legend: { display: false } }
      }
    });

    // 2. Gráfico de Plataformas (Donut)
    const pKeys = Object.keys(rawData.platform_share);
    const pValues = Object.values(rawData.platform_share);
    new Chart(document.getElementById('platformChart').getContext('2d'), {
      type: 'doughnut',
      data: {
        labels: pKeys,
        datasets: [{
          data: pValues,
          backgroundColor: ['#10b981', '#4f46e5'],
          borderWidth: 2
        }]
      },
      options: {
        responsive: true,
        maintainAspectRatio: false,
        plugins: { legend: { position: 'bottom' } }
      }
    });

    // 3. Gráfico de Países (Barra vertical)
    const countryKeys = Object.keys(rawData.country_share);
    const countryValues = Object.values(rawData.country_share);
    new Chart(document.getElementById('countryChart').getContext('2d'), {
      type: 'bar',
      data: {
        labels: countryKeys,
        datasets: [{
          data: countryValues,
          backgroundColor: '#3b82f6',
          borderRadius: 6
        }]
      },
      options: {
        responsive: true,
        maintainAspectRatio: false,
        plugins: { legend: { display: false } }
      }
    });

    // 4. Tabla de Productos
    const tbody = document.querySelector('#products-table tbody');
    rawData.productos.forEach(p => {
      const row = document.createElement('tr');
      row.innerHTML = `
        <td><strong>Producto #${p.producto_id}</strong></td>
        <td>${p.ventas.toLocaleString()}</td>
        <td>${p.ingresos.toLocaleString('es-ES', { minimumFractionDigits: 2 })} €</td>
      `;
      tbody.appendChild(row);
    });
  </script>
</body>
</html>
"""

# Reemplazar marcador de forma limpia sin usar f-strings complejos
final_html = html_template.replace("DATA_PLACEHOLDER", json.dumps(dashboard_payload))

# Renderizar
from IPython.display import HTML
HTML(final_html)


ID de Producto,Unidades Vendidas,Ingreso Acumulado


### 📊 Consulta Avanzada SQL: Análisis de ARPU por País y Canal
Esta consulta calcula el ingreso total, el número de compras y el ingreso promedio por transacción (ARPU por transacción) agrupado por la combinación de país y canal de marketing, excluyendo reembolsos.

In [13]:
# Consulta SQL que une usuarios y compras usando GROUP BY y filtros agregados corregidos
query_analisis_arpu = """
SELECT
    u.pais_limpio AS Pais,
    u.canal_limpio AS Canal_Adquisicion,
    COUNT(c.compra_id) AS Total_Ventas,
    ROUND(SUM(c.importe_numerico), 2) AS Ingresos_Totales_EUR,
    ROUND(AVG(c.importe_numerico), 2) AS Ticket_Promedio_EUR
FROM u_df u
INNER JOIN compras_validas c ON u.user_id = c.user_id
GROUP BY u.pais_limpio, u.canal_limpio
HAVING Ingresos_Totales_EUR > 100.0
ORDER BY Ingresos_Totales_EUR DESC;
"""

# Ejecutar consulta SQL sobre las variables locales
df_resultado_arpu = ps.sqldf(query_analisis_arpu, locals())

print("📈 Rendimiento de Ventas e Ingresos por Mercado y Canal (SQL):")
display(df_resultado_arpu)

📈 Rendimiento de Ventas e Ingresos por Mercado y Canal (SQL):


,Pais,Canal_Adquisicion,Total_Ventas,Ingresos_Totales_EUR,Ticket_Promedio_EUR
0,España,Facebook,1170,2869.81,2.45
1,España,Apple Search Ads,1182,2836.20,2.40
2,España,Google Ads,356,851.94,2.39
3,México,Orgánico,237,568.63,2.40
4,México,Google Ads,254,564.96,2.22
5,España,TikTok,195,531.05,2.72
6,Italia,Google Ads,194,454.56,2.34
7,Portugal,Facebook,114,335.86,2.95
8,Portugal,Google Ads,106,296.95,2.83
9,MX,Facebook,111,243.89,2.20


In [14]:
# 1. Crear diccionarios de mapeo oficiales para estandarización
mapeo_paises = {
    'es': 'España', 'españa': 'España', 'espania': 'España',
    'mx': 'México', 'mexico': 'México', 'méxico': 'México',
    'pt': 'Portugal', 'portugal': 'Portugal',
    'it': 'Italia', 'italia': 'Italia'
}

mapeo_canales = {
    'facebook': 'Facebook', 'meta': 'Facebook',
    'apple_search': 'Apple Search Ads', 'apple search': 'Apple Search Ads', 'asa': 'Apple Search Ads',
    'google': 'Google Ads', 'google_ads': 'Google Ads', 'google ads': 'Google Ads',
    'tiktok': 'TikTok', 'tik_tok': 'TikTok',
    'coinrush': 'Coin Rush', 'coin_rush': 'Coin Rush',
    'organico': 'Orgánico', 'organic': 'Orgánico', 'orgánico': 'Orgánico'
}

# 2. Aplicar la limpieza permanente sobre df_usuarios
df_usuarios['pais_limpio'] = df_usuarios['pais'].astype(str).str.strip().str.lower().map(mapeo_paises).fillna(df_usuarios['pais'].str.upper())
df_usuarios['canal_limpio'] = df_usuarios['canal_adquisicion'].astype(str).str.strip().str.lower().map(mapeo_canales).fillna('Otros')
df_usuarios['plataforma_limpia'] = df_usuarios['plataforma'].astype(str).str.strip().str.lower().replace({'ios': 'iOS', 'android': 'Android', 'and': 'Android'})

# 3. Actualizar las referencias que usa SQL para asegurar consistencia total
u_df = df_usuarios
compras_validas = df_compras_limpio[df_compras_limpio['reembolso'] == 0].copy()
# Asegurar que compras_validas tenga pegados los datos del usuario limpios
compras_validas = pd.merge(compras_validas, df_usuarios[['user_id', 'pais_limpio', 'canal_limpio', 'plataforma_limpia']], on='user_id', how='inner')

# 4. Volver a ejecutar la consulta SQL con los datos ya homogeneizados de raíz
query_analisis_arpu_limpio = """
SELECT
    c.pais_limpio AS Pais,
    c.canal_limpio AS Canal_Adquisicion,
    COUNT(c.compra_id) AS Total_Ventas,
    ROUND(SUM(c.importe_numerico), 2) AS Ingresos_Totales_EUR,
    ROUND(AVG(c.importe_numerico), 2) AS Ticket_Promedio_EUR
FROM compras_validas c
GROUP BY Pais, Canal_Adquisicion
HAVING Ingresos_Totales_EUR > 100.0
ORDER BY Ingresos_Totales_EUR DESC;
"""

df_resultado_arpu_limpio = ps.sqldf(query_analisis_arpu_limpio, locals())
print("✅ ¡Datos unificados con éxito! Aquí tienes el resultado limpio sin duplicaciones de países o canales:")
display(df_resultado_arpu_limpio)

✅ ¡Datos unificados con éxito! Aquí tienes el resultado limpio sin duplicaciones de países o canales:


,Pais,Canal_Adquisicion,Total_Ventas,Ingresos_Totales_EUR,Ticket_Promedio_EUR
0,España,Facebook,1170,2869.81,2.45
1,España,Apple Search Ads,1182,2836.20,2.40
2,España,Google Ads,356,851.94,2.39
3,México,Orgánico,265,626.86,2.37
4,México,Google Ads,275,613.25,2.23
5,España,TikTok,195,531.05,2.72
6,México,Facebook,219,463.32,2.13
7,Italia,Google Ads,194,454.56,2.34
8,Portugal,Facebook,114,335.86,2.95
9,Portugal,Google Ads,106,296.95,2.83


In [15]:
# @title 📊 Dashboard Avanzado PlayLoop (Datos 100% Limpios) {display-mode: "form"}

import json
import pandas as pd
from google.colab import output

# --- 1. Control de errores para JavaScript ---
def _report_js_error(message):
    print(f"JavaScript Error: {message}")
output.register_callback('report_js_error', _report_js_error)

# --- 2. Procesamiento de Datos de Variables Consolidadas ---
# Usamos los DataFrames limpios globales para evitar residuos
u_clean = df_usuarios.copy()
s_clean = df_sesiones_analisis.copy()

# Estandarizar plataformas unificadas
u_clean['plataforma_final'] = u_clean['plataforma_limpia']
s_clean['plataforma_final'] = s_clean['plataforma_limpia']
compras_validas['plataforma_final'] = compras_validas['plataforma_limpia']

# Obtener KPIs Generales
total_usuarios = int(u_clean['user_id'].nunique())
total_sesiones = int(s_clean['evento_id'].nunique())
total_ingresos = float(compras_validas['importe_numerico'].sum())
duracion_promedio = float(s_clean['duracion_segundos'].mean())

# Agregaciones
platform_distribution = u_clean['plataforma_final'].value_counts().to_dict()
ingresos_por_canal = compras_validas.groupby('canal_limpio')['importe_numerico'].sum().sort_values(ascending=False).to_dict()
country_distribution = u_clean['pais_limpio'].value_counts().head(5).to_dict()

# Productos IAP
productos_metricas = compras_validas.groupby('producto_id').agg(
    ventas=('compra_id', 'count'),
    ingresos=('importe_numerico', 'sum')
).reset_index()
productos_metricas['ingresos'] = productos_metricas['ingresos'].round(2)
productos_list = productos_metricas.sort_values(by='ingresos', ascending=False).to_dict(orient='records')

# Payload JSON para el Frontend
dashboard_payload = {
    "total_users": total_usuarios,
    "total_sessions": total_sessions,
    "total_revenue": round(total_ingresos, 2),
    "avg_session_duration": round(duracion_promedio, 1),
    "ingresos_por_canal": ingresos_por_canal,
    "platform_share": platform_distribution,
    "country_share": country_distribution,
    "productos": productos_list
}

# --- 3. Interfaz Web (HTML, CSS y JS) ---
html_template = """
<!DOCTYPE html>
<html lang="es">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>PlayLoop Clean Analytics</title>
  <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
  <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.4.0/css/all.min.css">
  <style>
    :root {
      --bg-color: #f8fafc;
      --card-bg: #ffffff;
      --primary: #6366f1;
      --primary-hover: #4f46e5;
      --success: #10b981;
      --warning: #f59e0b;
      --text-main: #0f172a;
      --text-muted: #64748b;
      --border-color: #e2e8f0;
    }
    * { box-sizing: border-box; margin: 0; padding: 0; }
    body {
      font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif;
      background-color: var(--bg-color);
      color: var(--text-main);
      padding: 20px;
    }
    .header { margin-bottom: 24px; border-bottom: 2px solid var(--border-color); padding-bottom: 16px; }
    .header h1 { font-size: 26px; color: var(--primary-hover); display: flex; align-items: center; gap: 12px; }
    .header p { font-size: 14px; color: var(--text-muted); margin-top: 4px; }

    /* KPI Row */
    .kpi-grid {
      display: grid;
      grid-template-columns: repeat(auto-fit, minmax(220px, 1fr));
      gap: 20px;
      margin-bottom: 24px;
    }
    .kpi-card {
      background: var(--card-bg);
      border-radius: 16px;
      padding: 24px;
      border: 1px solid var(--border-color);
      box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.05);
      display: flex;
      align-items: center;
      gap: 20px;
    }
    .kpi-icon {
      width: 54px; height: 54px; border-radius: 12px;
      display: flex; align-items: center; justify-content: center; font-size: 24px;
    }
    .blue-icon { background: #f0fdf4; color: #16a34a; }
    .green-icon { background: #eff6ff; color: #2563eb; }
    .purple-icon { background: #faf5ff; color: #7c3aed; }
    .amber-icon { background: #fffbeb; color: #d97706; }

    .kpi-info h3 { font-size: 12px; color: var(--text-muted); text-transform: uppercase; letter-spacing: 0.05em; margin-bottom: 4px; }
    .kpi-info .value { font-size: 24px; font-weight: 800; }

    /* Charts Layout */
    .charts-grid {
      display: grid;
      grid-template-columns: 1.5fr 1fr;
      gap: 24px;
      margin-bottom: 24px;
    }
    @media (max-width: 992px) { .charts-grid { grid-template-columns: 1fr; } }

    .chart-card {
      background: var(--card-bg);
      border-radius: 16px;
      padding: 24px;
      border: 1px solid var(--border-color);
      box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.05);
      display: flex;
      flex-direction: column;
    }
    .chart-card h2 { font-size: 16px; font-weight: 700; margin-bottom: 16px; display: flex; align-items: center; gap: 8px; }

    /* Wrapper to prevent iframe infinite heights */
    .canvas-wrapper {
      position: relative;
      flex-grow: 1;
      min-height: 280px;
      width: 100%;
    }

    /* Data Tables */
    .table-card {
      background: var(--card-bg);
      border-radius: 16px;
      padding: 24px;
      border: 1px solid var(--border-color);
      box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.05);
    }
    .table-card h2 { font-size: 16px; font-weight: 700; margin-bottom: 16px; }
    table { width: 100%; border-collapse: collapse; text-align: left; }
    th, td { padding: 14px 16px; border-bottom: 1px solid var(--border-color); font-size: 14px; }
    th { background-color: #f8fafc; font-weight: 700; color: var(--text-muted); text-transform: uppercase; font-size: 12px; }
    tr:hover { background-color: rgba(99, 102, 241, 0.03); }
  </style>
</head>
<body>

  <div class="header">
    <h1><i class="fa-solid fa-square-poll-vertical"></i> Dashboard de Operaciones Saneado (PlayLoop)</h1>
    <p>Todos los indicadores calculados con unificación geográfica completa y exclusión activa de devoluciones.</p>
  </div>

  <!-- KPI Summary Row -->
  <div class="kpi-grid">
    <div class="kpi-card">
      <div class="kpi-icon green-icon"><i class="fa-solid fa-users"></i></div>
      <div class="kpi-info">
        <h3>Usuarios Activos</h3>
        <div class="value" id="kpi-users">0</div>
      </div>
    </div>
    <div class="kpi-card">
      <div class="kpi-icon blue-icon"><i class="fa-solid fa-circle-dollar-to-slot"></i></div>
      <div class="kpi-info">
        <h3>Ingresos IAP Netos</h3>
        <div class="value" id="kpi-revenue">0.00 €</div>
      </div>
    </div>
    <div class="kpi-card">
      <div class="kpi-icon purple-icon"><i class="fa-solid fa-gamepad"></i></div>
      <div class="kpi-info">
        <h3>Sesiones Jugadas</h3>
        <div class="value" id="kpi-sessions">0</div>
      </div>
    </div>
    <div class="kpi-card">
      <div class="kpi-icon amber-icon"><i class="fa-solid fa-clock"></i></div>
      <div class="kpi-info">
        <h3>Duración Promedio</h3>
        <div class="value" id="kpi-duration">0s</div>
      </div>
    </div>
  </div>

  <!-- Charts Layout 1 -->
  <div class="charts-grid">
    <div class="chart-card">
      <h2><i class="fa-solid fa-chart-simple"></i> Ingresos por Canal de Adquisición Corregido</h2>
      <div class="canvas-wrapper">
        <canvas id="canalChart"></canvas></div>
    </div>
    <div class="chart-card">
      <h2><i class="fa-solid fa-mobile"></i> Distribución por Plataforma</h2>
      <div class="canvas-wrapper">
        <canvas id="platformChart"></canvas></div>
    </div>
  </div>

  <!-- Charts Layout 2 -->
  <div class="charts-grid">
    <div class="chart-card">
      <h2><i class="fa-solid fa-flag"></i> Top 5 Mercados de Adquisición</h2>
      <div class="canvas-wrapper">
        <canvas id="countryChart"></canvas></div>
    </div>
    <div class="chart-card table-card">
      <h2><i class="fa-solid fa-ranking-star"></i> Productos Más Vendidos</h2>
      <div style="overflow-y: auto; max-height: 280px;">
        <table id="products-table">
          <thead>
            <tr>
              <th>Producto</th>
              <th>Ventas</th>
              <th>Ingresos (€)</th>
            </tr>
          </thead>
          <tbody></tbody>
        </table>
      </div>
    </div>
  </div>

  <script>
    window.onerror = function(message) {
        google.colab.kernel.invokeFunction('report_js_error', [message], {});
    };

    const rawData = DATA_PLACEHOLDER;

    // Inyección de Métricas KPI
    document.getElementById('kpi-users').innerText = rawData.total_users.toLocaleString('es-ES');
    document.getElementById('kpi-revenue').innerText = rawData.total_revenue.toLocaleString('es-ES', { minimumFractionDigits: 2, maximumFractionDigits: 2 }) + ' €';
    document.getElementById('kpi-sessions').innerText = rawData.total_sessions.toLocaleString('es-ES');
    document.getElementById('kpi-duration').innerText = Math.round(rawData.avg_session_duration / 60) + ' min';

    // 1. Gráfico de Barra Horizontal: Canales
    const cKeys = Object.keys(rawData.ingresos_por_canal);
    const cValues = Object.values(rawData.ingresos_por_canal);
    new Chart(document.getElementById('canalChart').getContext('2d'), {
      type: 'bar',
      data: {
        labels: cKeys,
        datasets: [{
          label: 'Ingresos Netos (€)',
          data: cValues,
          backgroundColor: '#6366f1',
          borderRadius: 8,
          borderWidth: 0
        }]
      },
      options: {
        responsive: true,
        maintainAspectRatio: false,
        indexAxis: 'y',
        plugins: { legend: { display: false } },
        scales: {
          x: { grid: { display: false } },
          y: { grid: { display: false } }
        }
      }
    });

    // 2. Gráfico de Plataformas (Donut)
    const pKeys = Object.keys(rawData.platform_share);
    const pValues = Object.values(rawData.platform_share);
    new Chart(document.getElementById('platformChart').getContext('2d'), {
      type: 'doughnut',
      data: {
        labels: pKeys,
        datasets: [{
          data: pValues,
          backgroundColor: ['#10b981', '#6366f1'],
          borderWidth: 4,
          borderColor: '#ffffff'
        }]
      },
      options: {
        responsive: true,
        maintainAspectRatio: false,
        plugins: { legend: { position: 'bottom' } }
      }
    });

    // 3. Gráfico de Barra Vertical: Países Limpios
    const countryKeys = Object.keys(rawData.country_share);
    const countryValues = Object.values(rawData.country_share);
    new Chart(document.getElementById('countryChart').getContext('2d'), {
      type: 'bar',
      data: {
        labels: countryKeys,
        datasets: [{
          data: countryValues,
          backgroundColor: '#3b82f6',
          borderRadius: 8
        }]
      },
      options: {
        responsive: true,
        maintainAspectRatio: false,
        plugins: { legend: { display: false } },
        scales: {
          x: { grid: { display: false } },
          y: { grid: { display: false } }
        }
      }
    });

    // 4. Inyección de Tabla de Productos
    const tbody = document.querySelector('#products-table tbody');
    rawData.productos.forEach(p => {
      const row = document.createElement('tr');
      row.innerHTML = `
        <td><strong>Producto #${p.producto_id}</strong></td>
        <td>${p.ventas.toLocaleString('es-ES')}</td>
        <td>${p.ingresos.toLocaleString('es-ES', { minimumFractionDigits: 2, maximumFractionDigits: 2 })} €</td>
      `;
      tbody.appendChild(row);
    });
  </script>
</body>
</html>
"""

# Inyección limpia del JSON
final_html = html_template.replace("DATA_PLACEHOLDER", json.dumps(dashboard_payload))

# Renderizar interfaz de usuario
from IPython.display import HTML
HTML(final_html)

Producto,Ventas,Ingresos (€)


### 📤 Paso 6: Exportar Resultados a Google Sheets
Ejecuta la siguiente celda para autenticarte con tu cuenta de Google y guardar los DataFrames procesados directamente en una nueva hoja de cálculo en tu Google Drive.

In [16]:
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)

# 1. Crear el nuevo archivo de Google Sheets
sheet_title = "PlayLoop_Analytics_Report"
try:
    # Intentamos crear el archivo
    sh = gc.create(sheet_title)
    print(f"✅ Archivo '{sheet_title}' creado con éxito en tu Google Drive.")
except Exception as e:
    print(f"Ocurrió un error al crear el archivo: {e}")

# 2. Función auxiliar para exportar un DataFrame a una pestaña específica
def export_df_to_sheet(spreadsheet, dataframe, sheet_name):
    try:
        # Intentar abrir la pestaña si ya existe, si no, crearla
        try:
            worksheet = spreadsheet.worksheet(sheet_name)
        except gspread.exceptions.WorksheetNotFound:
            worksheet = spreadsheet.add_worksheet(title=sheet_name, rows="100", cols="20")

        # Limpiar contenido anterior y actualizar con el nuevo DataFrame
        worksheet.clear()
        worksheet.update([dataframe.columns.values.tolist()] + dataframe.values.tolist())
        print(f"  📊 Datos exportados correctamente a la pestaña: '{sheet_name}'")
    except Exception as e:
        print(f"  ❌ Error al exportar la pestaña '{sheet_name}': {e}")

# 3. Exportar las tablas de datos clave
print("Comenzando la exportación de tablas...")

# Tabla 1: Rentabilidad por Canal
export_df_to_sheet(sh, df_rentabilidad, "Rentabilidad_Canales")

# Tabla 2: ARPU por País y Canal
export_df_to_sheet(sh, df_resultado_arpu_limpio, "ARPU_Pais_Canal")

# Tabla 3: Rendimiento de Productos IAP
export_df_to_sheet(sh, productos_metricas.sort_values(by='ingresos', ascending=False), "Rendimiento_Productos")

# Eliminar la pestaña 'Sheet1' que se crea por defecto si está vacía
try:
    default_sheet = sh.worksheet("Sheet1")
    sh.del_worksheet(default_sheet)
except Exception:
    pass

print(f"\n🔗 ¡Todo listo! Puedes abrir tu reporte aquí: https://docs.google.com/spreadsheets/d/{sh.id}")

✅ Archivo 'PlayLoop_Analytics_Report' creado con éxito en tu Google Drive.
Comenzando la exportación de tablas...
  📊 Datos exportados correctamente a la pestaña: 'Rentabilidad_Canales'
  📊 Datos exportados correctamente a la pestaña: 'ARPU_Pais_Canal'
  📊 Datos exportados correctamente a la pestaña: 'Rendimiento_Productos'

🔗 ¡Todo listo! Puedes abrir tu reporte aquí: https://docs.google.com/spreadsheets/d/16Ut29MlHvKks8AEdvlds25BC6VD3TxMQDovS2GFLCpc
